In [39]:
import pandas as pd

In [40]:
df = pd.read_csv("IMDB Dataset.csv")

In [41]:
df.shape

(50000, 2)

In [42]:
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [43]:
df.isnull().sum()

review       0
sentiment    0
dtype: int64

In [44]:
df.drop_duplicates(inplace=True)
df.shape

(49582, 2)

## Pre_Processing

In [45]:
df["review"] = df["review"].str.lower()

In [46]:
import re

In [47]:
def remove_urls(text):
    text = re.sub(r"http\S+" , "", text)  
    return text
    
df["review"] = df["review"].apply(remove_urls)

In [48]:
def remove_punctuations(text):
    text = re.sub(r"[^A-Za-z0-9\s]" , "", text) 
    return text

df["review"] = df["review"].apply(remove_punctuations)

In [49]:
df.head()

,review,sentiment
0,one of the other reviewers has mentioned that ...,positive
1,a wonderful little production br br the filmin...,positive
2,i thought this was a wonderful way to spend ti...,positive
3,basically theres a family where a little boy j...,negative
4,petter matteis love in the time of money is a ...,positive


In [50]:
def remove_html(text):
    text = re.sub(r"<.*?>" , "", text)
    return text

df["review"] = df["review"].apply(remove_html)

In [51]:
import nltk

nltk.download("punkt")
nltk.download("punkt_tab")
nltk.download("stopwords")

[nltk_data] Downloading package punkt to /Users/ritesh/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/ritesh/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/ritesh/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [52]:
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

In [53]:
def remove_stopwords(text):
    tokens = word_tokenize(text)

    stop_words = set(stopwords.words("english"))

    negation_words = {
        "not", "no", "nor", "never",
        "don't", "doesn't", "didn't",
        "won't", "wouldn't",
        "can't", "couldn't",
        "isn't", "aren't",
        "wasn't", "weren't",
        "haven't", "hasn't", "hadn't"
    }

    stop_words = stop_words - negation_words

    tokens = [word for word in tokens if word.lower() not in stop_words]

    return " ".join(tokens)

df["review"] = df["review"].apply(remove_stopwords)

In [54]:
df.head()

,review,sentiment
0,one reviewers mentioned watching 1 oz episode ...,positive
1,wonderful little production br br filming tech...,positive
2,thought wonderful way spend time hot summer we...,positive
3,basically theres family little boy jake thinks...,negative
4,petter matteis love time money visually stunni...,positive


In [55]:
print(remove_stopwords("The acting was not impressive"))
print(remove_stopwords("This movie was a waste of time"))
print(remove_stopwords("This movie was not good"))

acting not impressive
movie waste time
movie not good


In [56]:
from nltk.stem import PorterStemmer

In [57]:
def stemming(text):
    ps = PorterStemmer()
    stemmed_words = []

    tokens = word_tokenize(text)
    for token in tokens:
        stemmed_token = ps.stem(token)
        stemmed_words.append(stemmed_token)

    return " ".join(stemmed_words)

df["review"] = df["review"].apply(stemming)

In [58]:
df.head()

,review,sentiment
0,one review mention watch 1 oz episod youll hoo...,positive
1,wonder littl product br br film techniqu unass...,positive
2,thought wonder way spend time hot summer weeke...,positive
3,basic there famili littl boy jake think there ...,negative
4,petter mattei love time money visual stun film...,positive


In [59]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

df["sentiment"] = le.fit_transform(df["sentiment"])

In [60]:
y = df["sentiment"]

In [61]:
y

0        1
1        1
2        1
3        0
4        1
        ..
49995    1
49996    0
49997    0
49998    0
49999    0
Name: sentiment, Length: 49582, dtype: int64

In [62]:
df.head()

,review,sentiment
0,one review mention watch 1 oz episod youll hoo...,1
1,wonder littl product br br film techniqu unass...,1
2,thought wonder way spend time hot summer weeke...,1
3,basic there famili littl boy jake think there ...,0
4,petter mattei love time money visual stun film...,1


In [63]:
from sklearn.feature_extraction.text import TfidfVectorizer

tf = TfidfVectorizer(max_features=5000)

X = tf.fit_transform(df["review"])

## Datasets & Data-Loaders

In [64]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [65]:
X_train.shape

(39665, 5000)

In [66]:
X_test.shape

(9917, 5000)

In [67]:
import torch
from torch.utils.data import TensorDataset, DataLoader

In [68]:
X_train = X_train.toarray()
X_test = X_test.toarray()

In [69]:
train_set = TensorDataset(
    torch.from_numpy(X_train).float(),
    torch.from_numpy(y_train.values).float()
)

test_set = TensorDataset(
    torch.from_numpy(X_test).float(),
    torch.from_numpy(y_test.values).float()
)

In [70]:
train_loader = DataLoader(train_set, shuffle=True, batch_size=64)
test_loader = DataLoader(test_set, shuffle=True, batch_size=64)

## Build RNN

In [71]:
import torch.nn as nn
import torch.optim as optim

In [72]:
class RNN(nn.Module):
    def __init__(self, input_size, hidden_size=128, num_layers=1):
        super().__init__()

        self.hidden_size = hidden_size
        self.num_layers = num_layers

        # RNN layer
        self.rnn = nn.RNN(input_size, hidden_size, num_layers, batch_first=True)

        # fully connected layer
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        # optional => shape (num of layers, batch size, hidden size)
        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size)

        out, _ = self.rnn(x, h0) 
        # 1st value = hidden state of all the timesteps => (batch, seq_len, hidden size)
        # 2nd value = final hidden state of last timestep

        out = self.fc(out[:, -1, :])
        return out

In [73]:
input_size = X_train.shape[1]

model = RNN(input_size)

criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters())

## Training the RNN

In [74]:
epochs = 10

for epoch in range(epochs):
    model.train()

    for Xb, yb in train_loader:
        optimizer.zero_grad()

        Xb = Xb.unsqueeze(1) # add singleton direction
        
        outputs = model(Xb) # (batch_size, 1)

        outputs = torch.sigmoid(outputs.squeeze()) # (batch_size,) => probability

        loss = criterion(outputs, yb) # compute loss
        loss.backward() # backprop
        optimizer.step() # weights update

    print(f"epoch = {epoch+1}/{epochs} and loss = {loss.item()}")

epoch = 1/10 and loss = 0.20896250009536743
epoch = 2/10 and loss = 0.2716082036495209
epoch = 3/10 and loss = 0.3897664546966553
epoch = 4/10 and loss = 0.23937392234802246
epoch = 5/10 and loss = 0.10998871922492981
epoch = 6/10 and loss = 0.2369876354932785
epoch = 7/10 and loss = 0.3784189522266388
epoch = 8/10 and loss = 0.1614232212305069
epoch = 9/10 and loss = 0.09019094705581665
epoch = 10/10 and loss = 0.09898608922958374


## Evaluate

In [75]:
model.eval()

with torch.no_grad():
    correct_vals = 0
    tot_vals = 0
    
    for Xb, yb in test_loader:
        Xb = Xb.unsqueeze(1)

        outputs = model(Xb)
        predicted = (torch.sigmoid(outputs.squeeze()) > 0.5).float()

        tot_vals += yb.size(0)
        correct_vals += (predicted == yb).sum().item()

    print(f"accuracy = {correct_vals/tot_vals*100}")

accuracy = 87.37521427851165


In [76]:
import pickle

# Save TF-IDF vectorizer
with open("tfidf_vectorizer.pkl", "wb") as f:
    pickle.dump(tf, f)

# Save model weights
torch.save(model.state_dict(), "rnn_model.pth")

print("Saved: tfidf_vectorizer.pkl and rnn_model.pth")

Saved: tfidf_vectorizer.pkl and rnn_model.pth
